# Phase 1: Data Understanding & Exploratory Data Inspection
**Project:** AI-Ecommerce-Analytics-Dashboard  
**Dataset:** Amazon Sales Report  
**Purpose:** Perform comprehensive read-only data understanding, schema audit, missing value profiling, outlier detection, and anomaly identification before any data cleaning.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visualization styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Define data path
data_path = os.path.join('..', 'Data', 'Raw Data', 'amazon_sales.csv.csv')
print(f"Data path exists: {os.path.exists(data_path)}")

# Load dataset in read-only inspection mode
df = pd.read_csv(data_path, low_memory=False)
print(f"Successfully loaded dataset with shape: {df.shape}")

## 2. Dataset Dimensions, Sample Rows & Data Types

In [ ]:
# Display shape and column list
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("\nColumn List:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}. {col} (dtype: {df[col].dtype})")

In [ ]:
# First 5 sample rows
df.head()

In [ ]:
# Concise schema overview
df.info()

## 3. Missing Value Analysis
Identify missing values (null / NaN) across all columns, calculate counts and percentages, and profile patterns of missingness.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.astype(str),
    'Missing Count': missing_count,
    'Missing Percentage (%)': missing_pct.round(2)
}).reset_index(drop=True)

# Filter to columns with missing values
missing_summary = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
missing_summary

In [ ]:
# Visualizing Missing Values Percentage
if len(missing_summary) > 0:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=missing_summary, x='Missing Percentage (%)', y='Column Name', palette='mako')
    plt.title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Missing Percentage (%)')
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()

## 4. Duplicate Rows & Granularity Analysis
Check for complete row duplicates vs multi-item orders with the same `Order ID`.

In [ ]:
# Check exact full duplicate rows
full_duplicates = df.duplicated().sum()
print(f"Full exact duplicate rows across all 24 columns: {full_duplicates}")

# Check unique Order IDs and line items per order
unique_orders = df['Order ID'].nunique()
total_records = len(df)
order_item_counts = df['Order ID'].value_counts()

print(f"Total unique Order IDs: {unique_orders:,}")
print(f"Single-item orders: {(order_item_counts == 1).sum():,} ({(order_item_counts == 1).mean()*100:.2f}%)")
print(f"Multi-item orders: {(order_item_counts > 1).sum():,} ({(order_item_counts > 1).mean()*100:.2f}%)")
print(f"Max items in a single order: {order_item_counts.max()}")

## 5. Numerical Columns & Outlier Detection
Analyze summary statistics, distributions, and potential outliers for `Qty`, `Amount`, and `ship-postal-code`.

In [ ]:
# Summary statistics for numerical columns
df[['Qty', 'Amount', 'ship-postal-code']].describe().T

In [ ]:
# IQR-based outlier detection for Amount
amount_clean = df['Amount'].dropna()
q25 = amount_clean.quantile(0.25)
q75 = amount_clean.quantile(0.75)
iqr = q75 - q25
lower_bound = q25 - 1.5 * iqr
upper_bound = q75 + 1.5 * iqr

outliers_amount = amount_clean[(amount_clean < lower_bound) | (amount_clean > upper_bound)]
print(f"Amount Q1 (25%): ₹{q25:.2f}")
print(f"Amount Q3 (75%): ₹{q75:.2f}")
print(f"IQR: ₹{iqr:.2f}")
print(f"IQR Lower Bound: ₹{lower_bound:.2f}")
print(f"IQR Upper Bound: ₹{upper_bound:.2f}")
print(f"Amount Outliers Count: {len(outliers_amount):,} ({len(outliers_amount)/len(amount_clean)*100:.2f}% of valid amounts)")

In [ ]:
# Visualizing Quantity and Amount Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantity Distribution
df['Qty'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Distribution of Order Quantity (Qty)', fontweight='bold')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log') # Log scale to view rare bulk orders

# Amount Distribution
sns.histplot(df['Amount'].dropna(), bins=50, kde=True, ax=axes[1], color='indigo')
axes[1].set_title('Distribution of Order Amount (INR)', fontweight='bold')
axes[1].set_xlabel('Amount (₹)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Categorical Columns Profiling
Examine unique categories, frequency counts, and cardinality across core attributes.

In [ ]:
cat_columns = ['Category', 'Size', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Courier Status', 'B2B']

for col in cat_columns:
    print(f"\n{'='*20} {col} (Unique: {df[col].nunique()}) {'='*20}")
    vc = df[col].value_counts(dropna=False)
    pct = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    col_summary = pd.DataFrame({'Count': vc, 'Percentage (%)': pct})
    print(col_summary.head(10))

In [ ]:
# Visualizing Top Categories & Order Status
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Product Category
df['Category'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Product Category Distribution', fontweight='bold')
axes[0].set_xlabel('Number of Orders')
axes[0].invert_yaxis()

# Top Order Statuses
df['Status'].value_counts().head(6).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top Order Statuses', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Date & Temporal Analysis
Audit transaction date parsing, date ranges, and daily volume distribution.

In [ ]:
# Parse dates
parsed_dates = pd.to_datetime(df['Date'], format='%m-%d-%y', errors='coerce')
print(f"Total date entries: {len(parsed_dates):,}")
print(f"Unparseable dates: {parsed_dates.isnull().sum()}")
print(f"Minimum Order Date: {parsed_dates.min().strftime('%Y-%m-%d')}")
print(f"Maximum Order Date: {parsed_dates.max().strftime('%Y-%m-%d')}")
print(f"Date Span: {(parsed_dates.max() - parsed_dates.min()).days} days (Q2 2022)")

In [ ]:
# Daily order volume trend
daily_orders = parsed_dates.value_counts().sort_index()
plt.figure(figsize=(14, 4))
daily_orders.plot(color='darkcyan', lw=2)
plt.title('Daily Order Volume Trend (Q2 2022)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Order Count')
plt.tight_layout()
plt.show()

## 8. Inconsistent, Anomalous & Suspicious Data Audit
Detect data entry quirks, logical mismatches between columns, and normalization targets for Phase 2.

In [ ]:
# 1. Column name whitespace check
print("Column headers with whitespace anomalies:")
for col in df.columns:
    if col != col.strip():
        print(f"  Found whitespace in column: '{col}' -> Stripped: '{col.strip()}'")

# 2. Relationship between Qty == 0, Amount, and Order Status
qty_zero = df[df['Qty'] == 0]
print(f"\nRows with Qty == 0: {len(qty_zero):,}")
print("Status breakdown for Qty == 0:")
print(qty_zero['Status'].value_counts())
print("\nAmount summary for Qty == 0:")
print(qty_zero['Amount'].describe())

# 3. Relationship between Fulfilment and fulfilled-by
print("\nCross-tabulation: Fulfilment vs fulfilled-by:")
print(pd.crosstab(df['Fulfilment'], df['fulfilled-by'].fillna('Missing (Amazon FBA)')))

# 4. State variations / casing check
unique_states_raw = df['ship-state'].nunique()
unique_states_upper = df['ship-state'].dropna().str.upper().nunique()
print(f"\nRaw unique state strings: {unique_states_raw}")
print(f"Uppercase unique state strings: {unique_states_upper} (Indicates case variations)")

## 9. Key Findings & Readiness for Phase 2

### Summary of Observations:
1. **Dataset Size:** 128,975 rows and 24 columns, representing 120,378 distinct customer orders.
2. **Zero Full Duplicates:** All rows are unique line items.
3. **Missing Values Identified:**
   - `fulfilled-by` (69.55% missing): Structural, corresponds to Amazon FBA.
   - `promotion-ids` (38.11% missing): Indicates orders without promotional discounts.
   - `Unnamed: 22` (38.03% missing): Redundant artifact column with only `False` values.
   - `Amount` & `currency` (6.04% missing): Concentrated in cancelled orders (7,566 out of 7,795 nulls).
   - `Courier Status` (5.33% missing): Concentrated in cancelled/unshipped orders.
   - `ship-city`, `ship-state`, `ship-postal-code`, `ship-country` (33 rows / 0.03% missing): Minor missing location records.
4. **Categorical & Structural Inconsistencies:**
   - Column header `'Sales Channel '` contains trailing whitespace.
   - State names contain mixed case (`Gujarat` vs `GUJARAT`) and abbreviation aliases (`NL`, `AR`, `PB`, `RJ`, `APO`).
   - 5,136 cancelled rows with `Qty = 0` retain listed amounts that must be treated appropriately during revenue aggregation.

**Next Step:** Awaiting review before initiating **Phase 2: Data Cleaning & Preprocessing**.